# NeuroGolf 2026 — GPU ML Training Pipeline v1For each unsolved task, train tiny Conv networks (3 architectures, 3 seeds).Filter for: val_acc=100% AND cost < thisray's. Export valid ONNX.**Inputs needed:** competitions/neurogolf-2026 dataset, thisray's submission.zip uploaded as private dataset.**Estimated runtime:** 8-12 hrs on T4.**Expected output:** /kaggle/working/improvements.zip with 50-100 ONNX files cheaper than thisray.

In [ ]:
!pip install -q onnx==1.21.0 onnxruntime==1.24.4 onnx-tool==1.0.1 numpy==2.4.4 2>&1 | tail -3

In [ ]:
"""GPU ML training pipeline â€” train tiny Conv per task, export to ONNX.Filters for: val_acc=100% AND cost < thisray's. Designed for Kaggle T4 GPU.Usage on Kaggle:1. Upload neurogolf-2026 dataset2. Upload thisray submission as input3. Run cells sequentially4. Output: improvements zip + per-task ONNX files"""import os, sys, json, math, pathlib, zipfile, tracebackimport numpy as npimport torchimport torch.nn as nnimport torch.nn.functional as Fimport onnxfrom onnx import helper, TensorProto# Kaggle pathsDATA = '/kaggle/input/competitions/neurogolf-2026/'THISRAY = '/kaggle/input/thisray-submission/submission.zip'OUT = pathlib.Path('/kaggle/working/improvements')OUT.mkdir(exist_ok=True)device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")def load_task(tn):    with open(f'{DATA}task{tn:03d}.json') as f:        return json.load(f)def to_onehot(grid, max_dim=30):    """Convert grid to (10, 30, 30) one-hot float32."""    arr = np.zeros((10, max_dim, max_dim), dtype=np.float32)    h, w = len(grid), len(grid[0]) if grid else 0    if h > max_dim or w > max_dim: return None    for r in range(h):        for c in range(w):            color = grid[r][c]            if 0 <= color < 10:                arr[color, r, c] = 1.0    return arrdef to_grid(arr):    """Convert (10, 30, 30) one-hot back to grid."""    h, w = 30, 30    g = []    for r in range(h):        row = []        for c in range(w):            colors = [k for k in range(10) if arr[k, r, c] > 0]            row.append(colors[0] if len(colors) == 1 else (11 if colors else 10))        # Trim trailing 10s        while row and row[-1] == 10:            row.pop()        g.append(row)    while g and not g[-1]:        g.pop()    return gdef task_to_tensors(task):    Xs, Ys = [], []    pairs = task.get('train', []) + task.get('test', []) + task.get('arc-gen', [])    for p in pairs:        x = to_onehot(p['input'])        y = to_onehot(p['output'])        if x is None or y is None: continue        Xs.append(x)        Ys.append(y)    if not Xs: return None, None    return np.stack(Xs), np.stack(Ys)# Architecturesclass Conv1Layer(nn.Module):    def __init__(self, k=3, h=10):        super().__init__()        self.conv = nn.Conv2d(10, h, k, padding=k//2)        self.out = nn.Conv2d(h, 10, 1)    def forward(self, x):        x = F.relu(self.conv(x))        return self.out(x)class Conv2Layer(nn.Module):    def __init__(self, k=3, h=16):        super().__init__()        self.conv1 = nn.Conv2d(10, h, k, padding=k//2)        self.conv2 = nn.Conv2d(h, h, k, padding=k//2)        self.out = nn.Conv2d(h, 10, 1)    def forward(self, x):        x = F.relu(self.conv1(x))        x = F.relu(self.conv2(x))        return self.out(x)class Conv3Layer(nn.Module):    def __init__(self, k=3, h=16):        super().__init__()        self.conv1 = nn.Conv2d(10, h, k, padding=k//2)        self.conv2 = nn.Conv2d(h, h, k, padding=k//2)        self.conv3 = nn.Conv2d(h, h, k, padding=k//2)        self.out = nn.Conv2d(h, 10, 1)    def forward(self, x):        x = F.relu(self.conv1(x))        x = F.relu(self.conv2(x))        x = F.relu(self.conv3(x))        return self.out(x)def export_onnx(model, fname):    model.eval()    dummy = torch.zeros(1, 10, 30, 30, device=device)    torch.onnx.export(        model, dummy, fname,        input_names=['input'], output_names=['output'],        opset_version=10, do_constant_folding=True,    )def cost_of(fname):    """Approximate cost using onnx-tool."""    try:        import onnx_tool        m = onnx_tool.loadmodel(fname, {'verbose': False, 'constant_folding': True})        m.graph.shape_infer(None)        m.graph.profile()        if not m.graph.valid_profile: return None        from onnx import shape_inference        model = onnx.load(fname)        graph = shape_inference.infer_shapes(model, strict_mode=True).graph        memory = 0        for item in list(graph.input) + list(graph.value_info) + list(graph.output):            if not item.type.HasField('tensor_type'): continue            tt = item.type.tensor_type            if not tt.HasField('shape'): return None            n = 1            for d in tt.shape.dim:                if d.HasField('dim_param'): return None                if not d.HasField('dim_value'): return None                n *= d.dim_value            if item.name in ['input', 'output']: continue            memory += n * np.dtype(onnx.helper.tensor_dtype_to_np_dtype(tt.elem_type)).itemsize        macs = sum(m.graph.macs) if hasattr(m.graph.macs, '__iter__') else m.graph.macs        params = sum(m.graph.params) if hasattr(m.graph.params, '__iter__') else m.graph.params        return int(macs) + memory + int(params)    except: return Nonedef validate_onnx_path(fname, X, Y):    """Run ORT inference, check (out>0) matches one-hot Y."""    import onnxruntime as ort    try:        sess = ort.InferenceSession(fname, providers=['CPUExecutionProvider'])    except: return False    for x, y in zip(X, Y):        try:            inp = x[None].astype(np.float32)            out = sess.run(['output'], {'input': inp})[0]            pred = (out[0] > 0.0).astype(np.float32)            if not np.array_equal(pred, y):                return False        except: return False    return Truedef train_one(task, tn, arch_class, **arch_kwargs):    X, Y = task_to_tensors(task)    if X is None or len(X) < 2: return None    Xt = torch.from_numpy(X).to(device)    Yt = torch.from_numpy(Y).to(device)    # Train each seed; pick best    for seed in range(3):        torch.manual_seed(seed)        model = arch_class(**arch_kwargs).to(device)        opt = torch.optim.Adam(model.parameters(), lr=0.01)        for epoch in range(200):            model.train()            logits = model(Xt)            loss = F.binary_cross_entropy_with_logits(logits, Yt)            opt.zero_grad(); loss.backward(); opt.step()        # Eval        model.eval()        with torch.no_grad():            pred = (model(Xt) > 0).float()            ok = torch.equal(pred, Yt)        if ok:            tmp = f'/tmp/task{tn:03d}_arch{arch_class.__name__}.onnx'            export_onnx(model, tmp)            return tmp    return Nonedef get_thisray_costs():    costs = {}    with zipfile.ZipFile(THISRAY) as z:        for inf in z.infolist():            if not inf.filename.endswith('.onnx'): continue            try: tn = int(inf.filename.replace('task','').replace('.onnx',''))            except: continue            tmp = f'/tmp/thisray_{tn}.onnx'            with open(tmp, 'wb') as f: f.write(z.read(inf.filename))            c = cost_of(tmp)            costs[tn] = c            os.remove(tmp)    return costsdef main():    thisray_costs = get_thisray_costs()    print(f"thisray costs computed: {len(thisray_costs)}")    archs = [        (Conv1Layer, {'k': 1, 'h': 16}),        (Conv1Layer, {'k': 3, 'h': 16}),        (Conv2Layer, {'k': 3, 'h': 16}),        (Conv3Layer, {'k': 3, 'h': 16}),    ]    wins = []    for tn in range(1, 401):        try:            task = load_task(tn)            tc = thisray_costs.get(tn)            if tc is None: continue            best = None            best_cost = tc            for ac, kw in archs:                tmp = train_one(task, tn, ac, **kw)                if not tmp: continue                X, Y = task_to_tensors(task)                if not validate_onnx_path(tmp, X, Y):                    os.remove(tmp); continue                c = cost_of(tmp)                if c is None or c >= best_cost:                    os.remove(tmp); continue                if best: os.remove(best)                best = tmp                best_cost = c            if best:                final = OUT / f'task{tn:03d}.onnx'                final.write_bytes(open(best, 'rb').read())                os.remove(best)                gain = (25 - math.log(max(1, best_cost))) - (25 - math.log(max(1, tc)))                wins.append({'tn': tn, 'old_cost': tc, 'new_cost': best_cost, 'gain': gain})                print(f"  WIN task{tn:03d}: {tc} -> {best_cost} (+{gain:.2f} pts)")        except Exception as e:            print(f"  task{tn:03d}: error {e}")            continue    print(f"\nTotal wins: {len(wins)}")    print(f"Total gain: +{sum(w['gain'] for w in wins):.2f} pts")    with open('/kaggle/working/wins.json', 'w') as f:        json.dump(wins, f)if __name__ == '__main__':    main()

In [ ]:
# Zip up improvementsimport zipfile, pathlibout = pathlib.Path('/kaggle/working')imp = out / 'improvements'with zipfile.ZipFile(out / 'improvements.zip', 'w', zipfile.ZIP_DEFLATED) as z:    for f in imp.glob('task*.onnx'):        z.write(f, f.name)print(f"Zipped {len(list(imp.glob('task*.onnx')))} ONNX files")